* `crossmatch_TOI_with_GaiaDR3.ipynb` attempts to cross-match based on position and magnitude difference.
* here result of previous cross-match from exofop or vizier is used

Last run: 241115

In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

# TOI catalog

In [ ]:
from wakai.catalog import get_tois

tois = get_tois(clobber=False)
tois.shape

In [ ]:
tois.tail()

## cross-match 

# cross-match/query TOI info from exofop

* CAUTION: might hit rate limit set by exofop

In [ ]:
from wakai.catalog import query_tfop_data

tfop_info = query_tfop_data("TOI-4364")
tfop_info['basic_info']

In [ ]:
import math
import json
from urllib.request import urlopen
from tqdm import tqdm

def get_tfop_info(target_name: str) -> dict:
    base_url = "https://exofop.ipac.caltech.edu/tess"
    url = f"{base_url}/target.php?id={target_name.replace(' ','')}&json"
    response = urlopen(url)
    assert response.code == 200, "Failed to get data from ExoFOP-TESS"
    try:
        data_json = json.loads(response.read())
        return data_json
    except Exception:
        #raise ValueError(f"No TIC data found for {target_name}")
        return
        
def get_GaiaDR2_id_from_exofop(tfop_info):
    """Do not use swifter
    """
    if tfop_info:
        try:
            names = tfop_info['basic_info'].get('star_names').split(', ')
            gaia = [i for i in names if i[:4]=='Gaia']
            if len(gaia)>0:
                return int(gaia[0].split()[-1])
        except Exception as e:
            print(e)
            return None

def update_toi_GaiaDR2_exofop(tois):
    for i,row in tqdm(tois.iterrows()):
        if math.isnan(row['GaiaDR3_exofop']):
            tfop_info = get_tfop_info(f"TIC {row['TIC ID']}")
            tois.loc[i,'GaiaDR3_exofop'] = get_GaiaDR2_id_from_exofop(tfop_info)
    return tois

In [ ]:
tois = update_toi_GaiaDR2_exofop(tois)

# cross-match with vizier

* possibly limit is not as strict as in exofop server

In [ ]:
try:
    import swifter
except:
    !{sys.executable} -m pip install swifter==1.3.4
    import swifter
assert swifter.__version__ == "1.3.4"

In [ ]:
cols = ['ra_deg','dec_deg']
toi_coords = tois[cols]

## method 1

In [ ]:
import warnings
warnings.simplefilter("ignore", category="UnitsWarning")
from wakai.catalog import Target

def get_GaiaDR3_id_from_vizier1(df_coords, param='Gaia', key='I/355/gaiadr3'):
    """
    Get Gaia ID by specifying catalog source using `key`.
    Try also key='I/345/gaia2'.

    Does not necessarily have GaiaDR3 entry.
    """
    return df_coords.swifter.apply(lambda coord: Target(*coord, verbose=False)\
            .query_vizier_param(param)\
            .get(key), axis=1)

def update_toi_GaiaDR2_vizier(tois):
    for i,row in tqdm(tois.iterrows()):
        if math.isnan(row['GaiaDR3_vizier']):
            tois.loc[i,'GaiaDR3_vizier'] = Target(row.ra_deg, row.dec_deg, verbose=False)\
            .query_vizier_param(param)\
            .get(key)
    return tois

## method 2 (recommended)

In [ ]:
import warnings
warnings.simplefilter("ignore", category="UnitsWarning")
from wakai.catalog import Target

class TargetWithProcessing(Target):
    def process_target(self, param='GaiaDR3'):
        # Perform the Vizier query and process results
        try:
            res = self.query_vizier_param(param)
            if res:
                return set(res.values()).pop()
            return None
        except Exception as e:
            print(e)
            return None

def get_GaiaDR3_id_from_vizier2(df_coords):
    """
    Get Gaia ID without specifying catalog source.
    Just get the unique set from potentially multiple sources.

    Not good if TOI is unpublished.
    """
    return df_coords.swifter.apply(lambda coord: TargetWithProcessing(*coord, verbose=False)\
            .process_target(), axis=1)

def update_toi_GaiaDR2_vizier(tois):
    for i,row in tqdm(tois.iterrows()):
        if math.isnan(row['GaiaDR3_vizier']):
            gaiaid = TargetWithProcessing(row.ra_deg, row.dec_deg, verbose=False)\
                .process_target(param='GaiaDR3')
            # edit in place
            tois.loc[i,'GaiaDR3_vizier'] = gaiaid
    return tois

In [ ]:
tois = update_toi_GaiaDR2_vizier(tois)

In [ ]:
tois.to_csv('../data/TOIs.csv', index=False)

# Comparison

In [ ]:
import pandas as pd

tois = pd.read_csv('../data/TOIs.csv')
#mismatch
sum((tois['GaiaDR3_exofop']-tois['GaiaDR3_vizier'])!=0)

In [ ]:
import pandas as pd

matches = pd.read_csv('../data/TOIs_GaiaDR3_xmatch_position.csv')
matches.head()

In [ ]:
import numpy as np

np.sum(matches.GaiaDR3>0), np.sum(tois.GaiaDR3_vizier>0), np.sum(tois.GaiaDR3_exofop>0)

In [ ]:
import pandas as pd

tois_missing = pd.read_csv('../data/TOIs_GaiaDR3_xmatch_missing.csv')
tois_missing.shape